# Geração de Texto com o GPT2

Aplicações de Geração de Texto na linha de Processamento de Linguagem Natural são referenciadas como Modelos Causais (Causal Models) que recebem um texto como contexto/prompt e completam o texto, seguindo o modelo de linguagem utilizado. Como exemplo de uso desse tipo de modelo, nós vamos utilizar um modelo pré-treinado do [GPT2](https://huggingface.co/pierreguillou/gpt2-small-portuguese).

Em uma visão geral, nesse caderno, nós vamos implementar as seguintes atividades:
1. Carregar um modelo pré-treinado de transformers (especificamente o GPT2);
2. Refinar/Treinar o modelo de linguagem com um córpus composto por regulamentos da UTFPR;
3. Avaliar o modelo gerado.

Para implementar essa aplicação, vamos inicialmente carregar uma dependência do projeto que será necessária para fazer o treinamento do modelo de Linguagem Causal.

```sh
pip install datasets
```

## Carregamento do modelo pré-treinado do GPT2
Após instalarmos as dependências da aplicação, vamos fazer o carregamento dos componentes Tokenizer e Modelo de Linguagem Causal. Ambos os componentes (Tokenizer e Modelo de Linguagem) são carregados separadamente, seguindo a API da biblioteca Transformers. O Tokenizer tem como objetivo gerar sequências de tokens a partir de textos, enquanto o modelo de linguagem recebe como entrada essa sequência e executa a aplicação especificada.

Nesse exemplo, observem que estamos utilizando a classe TFAutoModelForCausalLM. Em se tratando de uma classe CausalLM estamos especificando que vamos implementar uma aplicação de Geração de Texto. Pelo fato de utilizarmos uma classe que se inicia com a sigla TF, estamos especificando que o back-end desse modelo utilizará a API do TensorFlow.

O carregamento de ambos os componentes é realizado com a chamada from_pretrained e faz o download do: Tokenizer com o vocabulário já definido; e o modelo de linguagem pré-treinados.

In [ ]:
from transformers import AutoTokenizer, TFAutoModelForCausalLM


tokenizer = AutoTokenizer.from_pretrained("pierreguillou/gpt2-small-portuguese")
model = TFAutoModelForCausalLM.from_pretrained("pierreguillou/gpt2-small-portuguese")



/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/92.0 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/850k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/508k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

tf_model.h5:   0%|          | 0.00/498M [00:00<?, ?B/s]

All model checkpoint layers were used when initializing TFGPT2LMHeadModel.

All the layers of TFGPT2LMHeadModel were initialized from the model checkpoint at pierreguillou/gpt2-small-portuguese.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFGPT2LMHeadModel for predictions without further training.


Nessa etapa, nós já podemos fazer uso da estratégia de Transfer Learning. Esses componentes já foram pré-treinados e por isso já permitem a sua aplicação para geração de texto.

Para testar esse cenário, nós precisamos inicialmente gerar as sequências com o Tokenizer. Observem o uso do parâmetro return_tensors, identificando que vamos utilizar o back-end TensorFlow, e o atributo input_ids das sequências que será utilizado como entrada para a geração de textos.

Em seguida, passamos essa sequência como entrada para a chamada generate do modelo de linguagem. Essa chamada é específica para modelo de linguagem causais e gera sequências de tokens, considerando a distribuição de probabilidade de ocorrência de tokens, dada a sequência de entrada.

O resultado da chamada do método generate também é uma sequência de tokens. Para verificar o texto gerado, nós podemos utilizar o método decode do Tokenizer, que gera o texto a partir da sequência de tokens (no sentido inverso que utilizamos anteriormente).

In [ ]:
input = 'O cachorro gosta de andar na rua e '
model_inputs = tokenizer([input], return_tensors="tf").input_ids
print(model_inputs)

generated_text = model.generate(input_ids=model_inputs)
print(generated_text)
print(tokenizer.decode(generated_text[0], skip_special_tokens=True))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
/usr/local/lib/python3.10/dist-packages/transformers/generation/tf_utils.py:837: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length.  recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


tf.Tensor([[   47 21855  6249   261  8302   347  4823   258   221]], shape=(1, 9), dtype=int32)
tf.Tensor(
[[   47 21855  6249   261  8302   347  4823   258   221   199    65   450
   2241  6249   261 29594   297   488    14   199]], shape=(1, 20), dtype=int32)
O cachorro gosta de andar na rua e 
a sua mãe gosta de brincar com ele.



## Refinamento/Treinamento do modelo

Para refinar o modelo, nós podemos retreinar esse modelo com um córpus específico. Nesse exemplo, vamos utilizar como córpus, os regulamentos da UTFPR, em um cenário similar ao que vimos no conteúdo da semana passada.

Inicialmente, vamos fazer o download dos regulamentos em formato HTML.

In [ ]:
import io, tarfile, requests, os, pandas as pd


# download the dataset
def download (url, filename=''):
  if (os.path.isfile(filename)):
    print('Arquivo já existente no Runtime... Tudo OK')
    return
  response = requests.get(url)
  with open(f'./{filename}', 'wb') as f:
      f.write(response.content)
      print('Download realizado e arquivo extraído no Runtime... Tudo OK')

urls = [
  "https://raw.githubusercontent.com/watinha/nlp-text-mining-datasets/main/regulamentos/aa-ab-cf-dispensa-2021.html",
  "https://raw.githubusercontent.com/watinha/nlp-text-mining-datasets/main/regulamentos/ac-2022.html",
  "https://raw.githubusercontent.com/watinha/nlp-text-mining-datasets/main/regulamentos/diretrizes-grad-2022.html",
  "https://raw.githubusercontent.com/watinha/nlp-text-mining-datasets/main/regulamentos/ead-2022.html",
  "https://raw.githubusercontent.com/watinha/nlp-text-mining-datasets/main/regulamentos/estagio-2020.html",
  "https://raw.githubusercontent.com/watinha/nlp-text-mining-datasets/main/regulamentos/extensao-2022.html",
  "https://raw.githubusercontent.com/watinha/nlp-text-mining-datasets/main/regulamentos/rodp-2019.html",
  "https://raw.githubusercontent.com/watinha/nlp-text-mining-datasets/main/regulamentos/tcc-2022.html"
]

filenames = []

for url in urls:
  filename = url.split('/')[-1]
  filenames.append(filename)
  download(url, filename)



Download realizado e arquivo extraído no Runtime... Tudo OK
Download realizado e arquivo extraído no Runtime... Tudo OK
Download realizado e arquivo extraído no Runtime... Tudo OK
Download realizado e arquivo extraído no Runtime... Tudo OK
Download realizado e arquivo extraído no Runtime... Tudo OK
Download realizado e arquivo extraído no Runtime... Tudo OK
Download realizado e arquivo extraído no Runtime... Tudo OK
Download realizado e arquivo extraído no Runtime... Tudo OK


Após fazermos o download dos regulamentos em HTML, vamos extrair os dados textuais e limpar os dados, com estratégias similares ao que realizamos anteriormente na disciplina.

In [ ]:
import codecs

from bs4 import BeautifulSoup


corpus = []

for filename in filenames:
  with codecs.open(filename, encoding='cp1252') as f:
    html = f.read()
    soup = BeautifulSoup(html)
    ps = soup.select('div[unselectable=on] ~ p')
    article = ''

    for p in ps:
      if p.get_text().lower().startswith('art.'):
        article = p.get_text()
        corpus.append(article)
      else:
        paragraph = p.get_text()
        corpus.append(f'{article} {paragraph}')


print('\n - '.join([ doc[:100] for doc in corpus[:10]]))

 ANEXO DA RESOLUÇÃO COGEP/UTFPR Nº 110, DE 19 DE OUTUBRO DE 2021.
 -   
 -  Regulamento para 
as atividades acompanhadas, o abono de faltas, a compensação de faltas, 
a compen
 -   
 -   
 -  Capítulo I
 -  Das Atividades Acompanhadas
 - Art. 1º  As atividades acompanhadas 
caracterizam-se pela execução em condições específicas, de ativ
 - Art. 2º  Poderão solicitar a 
realização de atividades acompanhadas estudante regularmente matricula
 - Art. 2º  Poderão solicitar a 
realização de atividades acompanhadas estudante regularmente matricula


In [ ]:
def clean(doc):
  words = doc.split()
  chars_to_replace = '!"#$%&\'()*+,-:;<=>?@[\\]^_`{|}~'
  table = doc.maketrans(chars_to_replace, ' ' * len(chars_to_replace))
  cleaned_words = [w.translate(table) for w in words]
  cleaned_doc = ' '.join(cleaned_words)
  cleaned_doc = cleaned_doc.replace(u'\xa0', u' ')
  cleaned_doc = cleaned_doc.replace(u'\u200b', u' ')
  cleaned_doc = cleaned_doc.replace(u'\n', u' ')
  cleaned_doc = cleaned_doc.lstrip()

  return cleaned_doc


corpus = [ clean(doc) for doc in corpus ]
corpus = [ doc for doc in corpus if len(doc.lstrip()) > 0 ]
print('\n - '.join([ doc[:50] for doc in corpus[:10]]))



ANEXO DA RESOLUÇÃO COGEP/UTFPR Nº 110  DE 19 DE OU
 - 
 - Regulamento para as atividades acompanhadas  o abo
 - 
 - 
 - Capítulo I
 - Das Atividades Acompanhadas
 - Art. 1º As atividades acompanhadas caracterizam se
 - Art. 2º Poderão solicitar a realização de atividad
 - Art. 2º Poderão solicitar a realização de atividad


Tendo o córpus definido, nós precisamos gerar o dataset. A biblioteca Transformers utiliza uma estratégia própria para gerar o dataset de modelos de linguagem causais. Essa estratégia faz uso da API Datasets.

Com a definição do objeto Dataset, nós devemos fazer a tokenização do texto.

In [ ]:
from datasets import Dataset


dataset = Dataset.from_dict({ 'text': corpus })

print(dataset)

tokenized_dataset = dataset.map(lambda doc: tokenizer(
    doc['text'], padding=True, truncation=True, max_length=150, return_tensors="tf"), batched=True)
print(tokenized_dataset)

Dataset({
    features: ['text'],
    num_rows: 1179
})


Map:   0%|          | 0/1179 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 1179
})


Após gerarmos as sequências, nós podemos utilizar um componente DataCollator que deve gerar o dataset de treinamento do nosso modelo de linguagem.

In [ ]:
from transformers import DataCollatorForLanguageModeling


data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False, return_tensors="tf")

tf_train_set = model.prepare_tf_dataset(
    tokenized_dataset,
    shuffle=False,
    batch_size=16,
    collate_fn=data_collator,
)

print(tf_train_set)

<_PrefetchDataset element_spec=({'input_ids': TensorSpec(shape=(None, 150), dtype=tf.int64, name=None), 'attention_mask': TensorSpec(shape=(None, 150), dtype=tf.int64, name=None)}, TensorSpec(shape=(None, 150), dtype=tf.int64, name=None))>


Com o dataset, nós podemos retreinar o modelo para gerar um modelo de linguagem específico para os regulamentos da UTFPR utilizados. É importante observar que esse processo de treinamento pode levar um tempo razoável dependendo do hardware que está sendo utilizado.

In [ ]:
from transformers import AdamWeightDecay


model.compile(optimizer=AdamWeightDecay(learning_rate=2e-5, weight_decay_rate=0.01))
model.fit(x=tf_train_set, epochs=100)

Epoch 1/100
74/74 [==============================] - 51s 150ms/step - loss: 3.9625
Epoch 2/100
74/74 [==============================] - 11s 150ms/step - loss: 3.2747
Epoch 3/100
74/74 [==============================] - 11s 150ms/step - loss: 2.9262
Epoch 4/100
74/74 [==============================] - 11s 150ms/step - loss: 2.6547
Epoch 5/100
74/74 [==============================] - 11s 150ms/step - loss: 2.4364
Epoch 6/100
74/74 [==============================] - 11s 150ms/step - loss: 2.2508
Epoch 7/100
74/74 [==============================] - 11s 150ms/step - loss: 2.0929
Epoch 8/100
74/74 [==============================] - 11s 150ms/step - loss: 1.9545
Epoch 9/100
74/74 [==============================] - 11s 149ms/step - loss: 1.8277
Epoch 10/100
74/74 [==============================] - 11s 149ms/step - loss: 1.7176
Epoch 11/100
74/74 [==============================] - 11s 148ms/step - loss: 1.6161
Epoch 12/100
74/74 [==============================] - 11s 149ms/step - loss: 1.5312
E

Tendo concluído o treinamento, nós podemos avaliar os resultados, gerando textos com entradas definidas.

In [ ]:
prompts = ['Convalidação é um procedimento para que os alunos possam',
           'Como atividades completares podem ser realizadas',
           'O coeficiente é utilizado para determinar se o aluno',
           'Trabalho de Conclusão de Curso pode ser realizado no período',
           'Disciplinas optativas podem ser realizadas no momento em que']


for prompt in prompts:
  input_ids = tokenizer(prompt, return_tensors="tf").input_ids
  generated_text = model.generate(input_ids=input_ids, max_length=100)
  print('----------------------------------')
  print(f'  - {prompt}')
  print(generated_text)
  print(tokenizer.decode(generated_text[0], skip_special_tokens=True))
  print('\n')

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


----------------------------------
  - Convalidação é um procedimento para que os alunos possam
tf.Tensor(
[[ 4334   920  8931   372   342 11656   341   307   375  4429  8282   457
  42567   300  3579  1383 48173   300  4923  7629   261  1931  4494  3402
    261  2954  2961   221  3161 12647   414   468   221   325   921   414
    325  5176    14   384  2486   305 41544   420  4923   347  3899 49275
    298  3585   712   827 32201  2880  2956 10691   325  1275 13041   305
    491 12620  8301    14 29215   326  1068   315  3899 49275   298  3585
    307   468   827   300 13117   347   491 12620  8301   414   303  1486
    300  3056   261  8525  4569  2595   221  8475   457  4569  2595   347
   3899 49275   261  2086]], shape=(1, 100), dtype=int32)
Convalidação é um procedimento para que os alunos possam ser matriculados em unidades curriculares em cursos superiores de outra instituição pública de ensino superior  conveniadas ou não  no país ou no exterior. O resultado da validação dos c

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


----------------------------------
  - Como atividades completares podem ser realizadas
tf.Tensor(
[[ 3542  2773  5609   328  1239   457  5684   300   447   261   349   491
   9011   221   790   307   275   710   261  5266 12149   300  1181   491
   9011 12539   457  7480   325   337  8662   298  3585    14  8458  6330
   2328    14   384   710   261  5266 12149   300  1181   491  9011 12539
    457  7480   325   337  8662   298  3585    14 35139 15707    52 18443
     47  4724   221   221  2773   261  4819   221  2956  2364   391  2773
    261  4819    14  8458  6330  2328    14   384   710   261  5266 12149
    300  1181   491  9011 12539   457  7480   325   337  8662   298  3585
     14 35139 15707    52]], shape=(1, 100), dtype=int32)
Como atividades completares podem ser realizadas em mais de uma UCE  sendo que o tempo de atuação mínima em cada UCE deverá ser definido no PPC do curso. Parágrafo único. O tempo de atuação mínima em cada UCE deverá ser definido no PPC do curso. CAPÍT

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


----------------------------------
  - O coeficiente é utilizado para determinar se o aluno
tf.Tensor(
[[   47 21738   372  3519   341  7931   303   275  9507   827   300  5101
    261 37819   221   261  1445   297   377 11251   261 38370   221 14813
    367   475  3703   261 10971   221 21401   377  2642  9479   221  1943
    221   221 11175   414 13877   261  3579  1383 48173   221   325  2007
   3585  1484 29215   643  1068   384 21738   261 14266  5606  2340  2956
  26893   261  1445   297   259  7070   463   362   258  2956  3519   341
   6139   261 38370   258   300  1005  4620   307  1827   818   447   261
    342  3585    14  7070   463   643   221   529  2956  3519   341 45636
    261  5101   341  3579]], shape=(1, 100), dtype=int32)
O coeficiente é utilizado para determinar se o aluno está em risco de desligamento  de acordo com as instruções de matrícula  condicionada à existência de vagas  observadas as seguintes etapas  II   inclusão ou remoção de unidades curriculares  no

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


----------------------------------
  - Trabalho de Conclusão de Curso pode ser realizado no período
tf.Tensor(
[[22429   385   261  4232 21190   261 14034   838   457  3107   325  1345
  41633  5606   414   468   221  4902 22761 10164    14 29215    17  1068
    315 11855   298 10025   468   372  4070 34032   544  2486   305  7912
  22066   391   221  9813   414  4569  2595    14   575  1132  7655   275
  10025 22066   391   221  4569  2595   414  4569  2595   221  8475   457
  12747   391   221 10025 17155    14   575  1132  7655   275 10025 22066
    391   221 10025 17155   221   529  6793   261 10025 49275 17155   221
   8475   457 12747   391  3562   261  6929   261  3585   221   468 17155
    221 23165   457 41027]], shape=(1, 100), dtype=int32)
Trabalho de Conclusão de Curso pode ser realizado no período letivo normal ou não  conforme regulamentação específica. §1º A renovação do estágio não é efetivada pelo resultado da avaliação expresso como  aprovado ou reprovado. Em outras s

In [ ]:
model.save('./gpt2-regulamentos-utfpr-100ep.keras')

/usr/local/lib/python3.10/dist-packages/transformers/generation/tf_utils.py:465: UserWarning: `seed_generator` is deprecated and will be removed in a future version.
  warnings.warn("`seed_generator` is deprecated and will be removed in a future version.", UserWarning)
